# 07 · Is the data current?

The other notebooks ask *can I get the data, and is it right?* This one asks **is the data I got the right data for today?** A forecast can be complete and correct and still be the wrong one: an older run served after a newer delivery failed, or a run made on old months. Every answer then looks normal.

It makes five checks, from `/health` and the small country bulk file, and stops with a plain message if one fails:

| # | Question | How |
|---|---|---|
| 1 | Is the newest delivery the one served? | `/health`: a forecast is loaded, the service can reach its store, no newer delivery failed to load, no older forecast is served in its place; and it is the run you expect, if you set `CRAFD_EXPECTED_RUN` |
| 2 | How old is the forecast? | `/health`'s `forecast_freshness`, against the service's own limit |
| 3 | Does the first forecast month fit the forecast's date? | the `X-Forecast-Made-In` month must be the month of the forecast's date on `/health` (when the run was made, or else when it was delivered), or the month before; a run made on old months fails here. With check 2 this bounds how old the months can be; it does not compare them with today, because between deliveries a current forecast's first month is one or two months behind |
| 4 | Does the forecast start the month after the recorded deaths stop? | the months in the file: no gap, no overlap |
| 5 | How many forecast months? | counted in the file; checked if you set `CRAFD_EXPECTED_MONTHS` |

No number is written into this notebook: the limit comes from `/health`, and the run and the number of months are checked only against what you set.

**You need an API key** (set in `.env`, see `notebooks/README.md`).

## 1 · Setup

In [ ]:
import os
from datetime import datetime, timezone

import matplotlib.pyplot as plt
from dotenv import load_dotenv

from views_crafdapi.client import CrafdApiClient
from views_crafdapi.data_checks import (
    DataCheckError, check_forecast_age, check_forecast_month_fits_its_date, check_horizon, check_months,
    check_newest_delivery_served,
)

load_dotenv()  # reads CRAFD_KEY (your key) and, optionally, VIEWS_API_URL

BASE_URL = os.environ.get("VIEWS_API_URL", "https://crafdapi.viewsforecasting.org")
API_KEY = os.environ.get("CRAFD_KEY", "").strip().strip('"')
# Optional, for checking a release: the forecast run that should be served, and its number of
# forecast months. Unset, they are only shown.
EXPECTED_RUN = os.environ.get("CRAFD_EXPECTED_RUN") or None
try:
    EXPECTED_MONTHS = int(os.environ["CRAFD_EXPECTED_MONTHS"]) if os.environ.get("CRAFD_EXPECTED_MONTHS") else None
except ValueError:
    raise SystemExit("CRAFD_EXPECTED_MONTHS must be a whole number of months, such as 36.") from None
TODAY = datetime.now(timezone.utc).date()

client = CrafdApiClient(BASE_URL, API_KEY, timeout=300)
print("Base URL:", BASE_URL)
print("API key :", "set" if API_KEY else "MISSING: set it in .env")
print("Today   :", TODAY)

## 2 · Two quick calls

`/health`, and the country bulk file for one outcome (a few megabytes). If a check below fails, downloading more will not change the answer.

In [ ]:
if not API_KEY or API_KEY == "your-api-key-here":
    raise SystemExit("No usable API key. Set CRAFD_KEY in .env (see notebooks/README.md), restart "
                     "the kernel and run again.")
try:
    health = client.health()
except Exception as e:   # an error answer, a timeout or a dropped connection
    raise SystemExit(f"/health did not answer, so nothing here can be checked: {e}")

country, MADE_IN = client.download_bulk("country", "sb")
months = sorted(country["month"].astype(str).unique())
recorded = [m for m in months if m < MADE_IN]
forecast = [m for m in months if m >= MADE_IN]
freshness = health.get("forecast_freshness") or {}

print("Service status     :", health.get("status"))
print("Run served         :", health.get("served_run") or "not reported")
print("Forecast made      :", freshness.get("created_at"), "| age in days:", freshness.get("age_days"),
      "| limit:", freshness.get("sla_days"))
print("Forecast made in   :", MADE_IN, "(X-Forecast-Made-In)")
print("Recorded months    :", f"{recorded[0]} to {recorded[-1]} ({len(recorded)})" if recorded else "none")
print("Forecast months    :", f"{forecast[0]} to {forecast[-1]} ({len(forecast)})" if forecast else "none")

## 3 · The five checks

Each prints what it found. A `FAIL` is a finding about the data served, not about this notebook.

In [ ]:
def served_run_is_expected():
    served = health.get("served_run")
    if EXPECTED_RUN is None:
        return f"run {served or 'not reported'} (no CRAFD_EXPECTED_RUN set, so not checked)"
    if not served:
        raise DataCheckError("/health does not name the run served, so it cannot be checked")
    if served != EXPECTED_RUN:
        raise DataCheckError(f"the service serves {served}, not the expected {EXPECTED_RUN}")
    return f"run {served}, as expected"


checks = [
    ("1 · the newest delivery is served", lambda: check_newest_delivery_served(health)),
    ("1 · the run is the one expected", served_run_is_expected),
    ("2 · the forecast's age", lambda: check_forecast_age(health)),
    ("3 · the forecast's first month fits its date",
     lambda: check_forecast_month_fits_its_date(MADE_IN, (health.get("forecast_freshness") or {}).get("created_at"))),
    ("4 · the forecast starts after the recorded months", lambda: check_months(country, MADE_IN)),
    ("5 · the number of forecast months", lambda: check_horizon(country, MADE_IN, EXPECTED_MONTHS)),
]
failed = []
for name, run in checks:
    try:
        print(f"PASS  {name}: {run()}")
    except DataCheckError as problem:
        failed.append(name)
        print(f"FAIL  {name}: {problem}")
if failed:
    raise SystemExit(f"The data is not current: {len(failed)} check(s) failed: {', '.join(failed)}")
print("\nThe data is current.")

## 4 · The months on one line

Recorded and forecast months, with today marked. A gap or an overlap where they meet is visible here before you read a number.

In [ ]:
if not recorded or not forecast:
    raise SystemExit("The file has no recorded months or no forecast months, so there is nothing to draw.")


def as_year(month):
    return int(month[:4]) + (int(month[5:7]) - 1) / 12

fig, ax = plt.subplots(figsize=(11, 2.4))
if recorded:
    ax.barh(0, as_year(recorded[-1]) + 1 / 12 - as_year(recorded[0]), left=as_year(recorded[0]),
            height=0.5, color="#4c72b0", label=f"recorded deaths  {recorded[0]} to {recorded[-1]}")
if forecast:
    ax.barh(0, as_year(forecast[-1]) + 1 / 12 - as_year(forecast[0]), left=as_year(forecast[0]),
            height=0.5, color="#dd8452", label=f"forecast  {forecast[0]} to {forecast[-1]}")
today_x = TODAY.year + (TODAY.month - 1) / 12 + (TODAY.day - 1) / 365
ax.axvline(today_x, color="black", lw=1.2, ls="--")
ax.annotate("today", (today_x, 0.4), ha="center", fontsize=9)
ax.set_yticks([])
ax.set_ylim(-0.6, 0.7)
ax.set_xlim(as_year(forecast[0]) - 6, as_year(forecast[-1]) + 1)   # the last years of records, and the forecast
ax.set_xlabel("year")
ax.legend(loc="upper left", frameon=False, fontsize=9)
ax.set_title(f"Months served  (run {health.get('served_run') or 'not reported'})", loc="left")
plt.tight_layout()
plt.show()

## 5 · Does the forecast pick up where the recorded deaths stop?

The world's total, summed over countries: recorded deaths (`sb_actual`) in the last twelve recorded months, expected deaths (`sb_mean`) in the first twelve forecast months. They are different quantities and will not meet exactly. What to look for is that the forecast starts near where the records leave off; a jump by a factor of ten at the join points to a unit error or a wrong run.

The last recorded month is only partly recorded when the forecast is made (recorded deaths arrive about a month late), so it can sit low.

In [ ]:
if not recorded or not forecast:
    raise SystemExit("The file has no recorded months or no forecast months, so there is no join to draw.")
world = country.groupby("month")[["sb_actual", "sb_mean"]].sum(min_count=1)
recorded_tail = world.loc[recorded[-12:], "sb_actual"]
forecast_head = world.loc[forecast[:12], "sb_mean"]

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(recorded_tail.index, recorded_tail.values, marker="o", color="#4c72b0", label="recorded deaths (sb_actual)")
ax.plot(forecast_head.index, forecast_head.values, marker="o", color="#dd8452", label="expected deaths (sb_mean)")
ax.axvline(len(recorded_tail) - 0.5, color="black", ls="--", lw=1.0)
ax.set_ylabel("deaths in the world, per month")
ax.tick_params(axis="x", rotation=45)
ax.set_title("State-based deaths across the join", loc="left")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

print(f"Last recorded month  {recorded[-1]}: {recorded_tail.iloc[-1]:,.0f} deaths")
print(f"First forecast month {forecast[0]}: {forecast_head.iloc[0]:,.0f} expected deaths")

## 6 · What a failure means

| Check | If it fails |
|---|---|
| 1 | You are getting an older forecast than the newest delivered, with answers that look normal. Tell VIEWS. |
| 2 | The forecast is out of date: a new one is late. Tell VIEWS. |
| 3 | The forecast was made on old months, or the service does not say when it was made. Tell VIEWS. |
| 4 | Months are missing between the records and the forecast, or both cover the same month. Do not use the data; tell VIEWS. |
| 5 | The forecast is cut short. Do not use the months beyond it; tell VIEWS. |

Whether the numbers are plausible is for you to judge from the maps in `02_visualization.ipynb`; whether they are internally consistent, from section 4 of `01_quickstart.ipynb`.